# s15 — Parallel Tool Calls

**What this teaches:** when a model emits more than one function call in a single response, ADK dispatches them in **the order the model emitted them** — and they run truly in parallel under the hood. You do not need to write any scheduling code; you simply attach tools, prompt the model to use several at once, and the runner handles concurrency.

**Concept anchor:** the agent loop is *model → tool(s) → model*, where the middle step is a batch. If the model returns three `tool_call`s in one turn, three tools execute concurrently and the next model turn sees all three results merged back into the conversation.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- A provider whose model is comfortable emitting multiple tool calls per turn. Anthropic Sonnet/Opus and OpenAI GPT-4o are good defaults.
  ```
  export OMNIS_PROVIDER=openai_compat
  export OPENAI_BASE_URL=http://localhost:11434/v1
  export OMNIS_MODEL=qwen2.5-coder
  ```

## 1. Imports

We need the file-system toolset (`core/tools`) for the model to have something concrete to call in parallel.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/omnis/core/agentkit"
    "github.com/blouargant/omnis/core/stream"
    fstools "github.com/blouargant/omnis/core/tools"
)

## 2. Helper

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Build the model and agent

The key is the `Instruction` — we explicitly nudge the model toward issuing several tool calls in the same turn. The `Tools` field gets the full read-only file-system suite (`read`, `glob`, `grep`, `write`, `bash`, …).

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s15_parallel",
    Description: "Multiple tool calls in one turn.",
    Instruction: "When asked, call several read/glob/grep tools in the same turn.",
    Model:       llm,
    Tools:       fstools.New(),
})
must(err)

## 4. Build the runner

Nothing fancy — same minimal runner you saw in [s01_loop](../s01_loop/s01_loop.ipynb). The parallelism is **a property of the ADK dispatcher**, not of any plugin.

In [ ]:
r, err := agentkit.Runner("s15", a)
must(err)

## 5. Prompt the model to batch its calls

We ask for three logically independent operations in one sentence. A model that supports parallel tools will return all three calls in a single response; ADK then runs them concurrently and feeds the merged results into the next turn.

In [ ]:
prompt := "Use glob to list *.go files, then read the first one's first 5 lines, then grep for 'package' in the same file. Do them all in one turn if possible."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- In the streamed output you should see several `tool_call` entries followed by their `tool_result` entries interleaved — not three separate model turns. Compare with [s05_tools](../s05_tools/s05_tools.ipynb), where tool calls are typically sequential.
- Pair this notebook with [s18_events](../s18_events/s18_events.ipynb): the event counter will show multiple `before_tool` events sandwiched between a single pair of `before_model`/`after_model` events.

## Try it yourself

1. Drop the `Instruction` field and re-run — many models will fall back to sequential calls without the nudge.
2. Replace the prompt with one logical chain of dependent calls ("read X, then based on its content read Y") and confirm the model now issues calls turn-by-turn, since each one needs the previous result.